In [ ]:
import pandas as pd


df = pd.read_csv('/content/drive/MyDrive/DX PROJ./층간소음데이터통합_영어제거완료.csv')

In [ ]:
df

In [ ]:
# '본문' 컬럼에 NaN이 있는 행만 삭제
df.dropna(subset=['본문'], inplace=True)

In [ ]:
df

In [ ]:
!pip install sentence-transformers torch transformers

In [ ]:
import torch

# GPU가 연결되었는지 확인하는 코드
if torch.cuda.is_available():
    print(f"성공! 현재 사용 가능한 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU 연결 실패... CPU 모드입니다.")

In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# 모델 불러오기
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

print("모델 로드 완료!")

In [ ]:
# 1. '통합본문' 컬럼에 결측치가 있는 행을 제거합니다.
df = df.dropna(subset=['본문'])

# 2. (혹시 모르니) 모든 데이터를 문자열 타입으로 확실히 변환합니다.
text_list = df['본문'].astype(str).tolist()

# 3. 다시 임베딩을 진행합니다.
print(f"총 {len(text_list)}개의 문장을 임베딩합니다...")
embeddings = model.encode(text_list, show_progress_bar=True)

# 4. 결과 확인
print("임베딩 완료! 형태:", embeddings.shape)

In [ ]:
df

In [ ]:
# 1. 생성된 임베딩 벡터를 리스트 형태로 변환하여 새로운 'embedding' 컬럼에 할당
df['embedding'] = embeddings.tolist()

# 2. 결과 확인 (embedding 컬럼이 잘 들어갔는지 확인)
display(df.head())

In [ ]:
# (선택) 데이터프레임 저장 예시
df.to_pickle("embedded_data bert.pkl")

-----------------------------
여기부터 수정

In [ ]:
import pandas as pd
import pickle

# 판다스로 피클 파일 바로 읽기
df = pd.read_pickle("embedded_data bert.pkl")

print("✅ 데이터프레임 로드 완료!")
display(df.head())

In [ ]:
import ast
import numpy as np

# 1. 만약 컬럼명이 지정되어 있지 않다면, 4번째 컬럼(인덱스 3)의 이름을 'embedding'으로 지정해줍니다.
# (출력해주신 데이터 구조상 4번째 열에 벡터가 들어있기 때문입니다)
df.columns = ['제목', '본문', 'token', 'embedding']  # 필요에 따라 컬럼명을 맞춰주세요

# 2. [치트키] 문자열(str)로 인식된 벡터를 실제 파이썬 리스트(숫자) 형태로 변환합니다.
# 데이터가 6만 건이기 때문에, 타입이 문자열(str)인 경우에만 안전하게 숫자로 파싱해줍니다.
if isinstance(df['embedding'].iloc[0], str):
    print("🔄 문자열로 된 벡터를 숫자 배열로 변환 중입니다... (6만 건 기준 약 10~20초 소요)")
    df['embedding'] = df['embedding'].apply(ast.literal_eval)

# 3. [최종 단계] 리스트들의 시리즈를 UMAP/PCA가 좋아하는 거대한 하나의 넘파이 행렬로 결합합니다.
bert_embeddings = np.array(df['embedding'].tolist())

# 4. 성공적으로 만들어졌는지 모양새(Shape) 확인하기
print("✅ bert_embeddings 준비 완료!")
print(f"📊 Matrix Shape: {bert_embeddings.shape}")
# 정상적이라면 (60000, 1536) 혹은 (60000, 768) 형태로 행렬 크기가 출력됩니다.

### 150차원, 5차원

In [ ]:

from sklearn.decomposition import PCA
from umap import UMAP

# bert_embeddings.shape가 (60000, 768)인 상황 가정

# -------------------------------------------------------------
# [1단계] PCA로 거대한 차원의 노이즈를 빠르게 거르고 경량화 (768D -> 100D)
# -------------------------------------------------------------
print("🚀 1단계: PCA 차원 축소 시작...")
pca = PCA(n_components=150, random_state=42) # 100차원~150차원 추천
embeddings_pca = pca.fit_transform(bert_embeddings)
print(f"✅ PCA 완료! 배열 모양: {embeddings_pca.shape}")
# 결과: (60000, 100) -> 이제 컴퓨터가 숨을 쉴 수 있는 크기가 되었습니다.


# -------------------------------------------------------------
# [2단계] 가벼워진 PCA 배열을 UMAP에 넣어 정밀한 공간 구축 (100D -> 5D 또는 10D)
# -------------------------------------------------------------
print("\n🚀 2단계: UMAP 정밀 차원 축소 시작 (이웃 관계 계산)...")
# 6만 건이므로 n_neighbors를 15~30 정도로 여유 있게 주어 거대한 흐름을 잡습니다.
umap_model = UMAP(n_components=5, n_neighbors=30, min_dist=0.1, random_state=42)
embeddings_umap = umap_model.fit_transform(embeddings_pca)
print(f"✅ UMAP 완료! 최종 배열 모양: {embeddings_umap.shape}")


# -------------------------------------------------------------
# [3단계] 이 최종 고도화 배열을 가지고 최적의 K 사냥하러 가기
# -------------------------------------------------------------
# 이 embeddings_umap 배열을 가지고 이전에 구상하셨던
# 엘보우 기법(Elbow)과 실루엣 지수(Silhouette) 코드를 돌리시면 됩니다!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. 6만 건 데이터 중 실루엣 계산용 5,000건 샘플링 (속도 및 메모리 보호)
np.random.seed(42)
sample_size = 5000
sample_idx = np.random.choice(len(embeddings_umap), size=sample_size, replace=False)
umap_sample = embeddings_umap[sample_idx]

inertia_list = []
silhouette_list = []
k_range = range(2, 9)  # K = 2개부터 8개까지 방 개수 테스트

print("🔄 10차원 UMAP 공간 위에서 최적의 K 사냥 시작... (약 1분 소요)\n")

for k in k_range:
    # KMeans 모델 선언
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)

    # [엘보우] Inertia는 전체 데이터(6만 건)를 완벽히 반영하여 학습
    kmeans.fit(embeddings_umap)
    inertia_list.append(kmeans.inertia_)

    # [실루엣] 추출한 샘플 데이터(5,000건)로 예측 및 분리도 계산
    sample_labels = kmeans.predict(umap_sample)
    score = silhouette_score(umap_sample, sample_labels)
    silhouette_list.append(score)

    print(f"📊 [K = {k}] 연산 완료 | Inertia: {kmeans.inertia_:.2f} | Silhouette Score: {score:.4f}")

print("\n✅ 모든 연산이 완료되었습니다! 그래프를 그립니다.")

# 2. 이중 축(Dual-Axis) 시각화 그래프 그리기
fig, ax1 = plt.subplots(figsize=(11, 6))

# 엘보우 기법 (왼쪽 Y축 - 파란색 실선)
color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (K)', fontweight='bold', fontsize=12)
ax1.set_ylabel('Inertia (Elbow Method)', color=color, fontweight='bold', fontsize=12)
ax1.plot(k_range, inertia_list, marker='o', color=color, linewidth=2.5, label='Inertia')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

# 실루엣 지수 (오른쪽 Y축 - 주황색 점선)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Silhouette Score (Higher is Better)', color=color, fontweight='bold', fontsize=12)
ax2.plot(k_range, silhouette_list, marker='s', color=color, linewidth=2.5, linestyle='--', label='Silhouette')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Optimal K Search on UMAP 5D Space (60k docs)', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

# 1. 마음의 결정을 내린 최종 K값 설정
final_k = 4

print(f"🚀 확정된 K={final_k}로 최종 K-Means 군집화 가동...")
final_kmeans = KMeans(n_clusters=final_k, init='k-means++', random_state=42, n_init=10)

# 5차원 UMAP 배열을 이용해 정밀 학습 및 예측
# (앞 셀에서 만든 embeddings_umap을 그대로 사용합니다)
df['cluster'] = final_kmeans.fit_predict(embeddings_umap)

print("📊 각 군집별 데이터 배정 개수 확인:")
print(df['cluster'].value_counts())

# 2. [치트키] 이제 다음 단계(LDA, TF-IDF)를 위해 데이터프레임을 피클로 안전하게 저장
df.to_pickle("층간소음_군집완료_4.pkl")


In [ ]:
import numpy as np
from umap import UMAP
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 시각화용 2차원 투영 데이터 생성 (★10차원 전용 배열인 embeddings_umap_10d 로 변경 완료)
if embeddings_umap_10d.shape[1] > 2:
    print(f"🔄 현재 {embeddings_umap_10d.shape[1]}차원 배열이므로, 시각화를 위해 2차원 UMAP 플롯을 추가 생성합니다...")
    # 거대한 흐름을 안정적으로 유지하기 위해 n_neighbors=30 세팅 유지
    umap_2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42, n_jobs=-1)
    viz_data = umap_2d.fit_transform(embeddings_umap_10d)
else:
    viz_data = embeddings_umap_10d

# --- 시각화 그래프 그리기 ---
plt.figure(figsize=(12, 9))
sns.set_theme(style='whitegrid')

# ★ [수정 핵심 포인트] ★
# hue 대상을 10차원 전용 군집 레이블인 df['cluster_10d']로 완벽 매칭했습니다.
sns.scatterplot(
    x=viz_data[:, 0],
    y=viz_data[:, 1],
    hue=df['cluster_10d'],  # 10차원 K-Means 방 번호 컬럼 입력
    palette='Set2',         # 눈 피로도가 적고 이쁜 파스텔톤 컬러 테마
    alpha=0.5,              # 6만 건의 밀도를 정밀하게 보기 위한 투명도
    s=4,                    # 대용량 점들의 뭉침을 예쁘게 표현하는 최적 크기
    linewidth=0             # 점들의 검은 테두리를 제거하여 대륙 색상을 선명하게 보존
)

# 그래프 제목 및 축 이름 레이블링 (★10D 정보를 명시하여 혼선 방지)
plt.title(f"BERT + PCA + UMAP 10D Final Cluster Map (K={df['cluster_10d'].nunique()})", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("UMAP Dimension 1", fontsize=11)
plt.ylabel("UMAP Dimension 2", fontsize=11)

# 범례(Legend) 상자를 그래프 우측 바깥 영역으로 깔끔하게 격리
plt.legend(title="Cluster No.", bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=3)

# ★ 파일명도 5차원 결과와 덮어써지지 않게 'umap_10d_final_cluster_map.png'로 분리 저장
plt.savefig("umap_10d_final_cluster_map.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"🎉 축하합니다! {df['cluster_10d'].nunique()}개 방으로 선명하게 분할된 10차원 최종 색상 지도가 'umap_10d_final_cluster_map.png'로 저장되었습니다!")

In [ ]:
# 현재 메모리에 올라온 UMAP 배열의 크기 확인
print("📊 현재 배열의 모양(Shape):", embeddings_umap.shape)

### 150차원, 10차원

In [ ]:

from sklearn.decomposition import PCA
from umap import UMAP

# bert_embeddings.shape가 (60000, 768)인 상황 가정

# -------------------------------------------------------------
# [1단계] PCA로 거대한 차원의 노이즈를 빠르게 거르고 경량화 (768D -> 100D)
# -------------------------------------------------------------
print("🚀 1단계: PCA 차원 축소 시작...")
pca = PCA(n_components=150, random_state=42) # 100차원~150차원 추천
embeddings_pca = pca.fit_transform(bert_embeddings)
print(f"✅ PCA 완료! 배열 모양: {embeddings_pca.shape}")
# 결과: (60000, 100) -> 이제 컴퓨터가 숨을 쉴 수 있는 크기가 되었습니다.


# -------------------------------------------------------------
# [2단계] 가벼워진 PCA 배열을 UMAP에 넣어 정밀한 공간 구축 (100D -> 5D 또는 10D)
# -------------------------------------------------------------
print("\n🚀 2단계: UMAP 정밀 차원 축소 시작 (이웃 관계 계산)...")
# 6만 건이므로 n_neighbors를 15~30 정도로 여유 있게 주어 거대한 흐름을 잡습니다.
umap_model = UMAP(n_components=10, n_neighbors=30, min_dist=0.1, random_state=42)
embeddings_umap = umap_model.fit_transform(embeddings_pca)
print(f"✅ UMAP 완료! 최종 배열 모양: {embeddings_umap.shape}")


# -------------------------------------------------------------
# [3단계] 이 최종 고도화 배열을 가지고 최적의 K 사냥하러 가기
# -------------------------------------------------------------
# 이 embeddings_umap 배열을 가지고 이전에 구상하셨던
# 엘보우 기법(Elbow)과 실루엣 지수(Silhouette) 코드를 돌리시면 됩니다!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# =============================================================
# [STEP 1] 이름을 완전히 다르게 해서 10차원 UMAP 강제 새출발
# =============================================================
print("🚀 [강제 갱신] 1단계: 10차원 UMAP 압축 연산 시작합니다...")

umap_10d_forced = UMAP(n_components=10, n_neighbors=30, min_dist=0.1, random_state=42, n_jobs=-1)
# 꼬임 방지를 위해 변수명을 완전히 'embeddings_umap_10d'로 차별화합니다.
embeddings_umap_10d = umap_10d_forced.fit_transform(embeddings_pca)

print("📊 [검증] 현재 메모리에 생성된 배열의 실제 모양(Shape):")
print(embeddings_umap_10d.shape)
# -> 반드시 (60000, 10)이 찍혀야 정상입니다! 확인해보세요.


# =============================================================
# [STEP 2] 새로 태어난 10차원 배열(embeddings_umap_10d)로 곧바로 K 사냥
# =============================================================
print("\n🔄 2단계: 따끈따끈한 10차원 데이터로 엘보우/실루엣 재계산 중...")

np.random.seed(42)
sample_size = 5000
sample_idx = np.random.choice(len(embeddings_umap_10d), size=sample_size, replace=False)
umap_sample = embeddings_umap_10d[sample_idx]

inertia_list = []
silhouette_list = []
k_range = range(2, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)

    # 10차원 배열 적용
    kmeans.fit(embeddings_umap_10d)
    inertia_list.append(kmeans.inertia_)

    sample_labels = kmeans.predict(umap_sample)
    score = silhouette_score(umap_sample, sample_labels)
    silhouette_list.append(score)

    print(f"📊 [10D / K = {k}] Inertia: {kmeans.inertia_:.2f} | Silhouette: {score:.4f}")


# =============================================================
# [STEP 3] 10차원 전용 이중 축 그래프 시각화
# =============================================================
print("\n🎨 3단계: 10차원 전용 성적표 그래프를 그립니다.")
fig, ax1 = plt.subplots(figsize=(11, 6))

color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (K)', fontweight='bold', fontsize=12)
ax1.set_ylabel('Inertia (Elbow Method)', color=color, fontweight='bold', fontsize=12)
ax1.plot(k_range, inertia_list, marker='o', color=color, linewidth=2.5)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Silhouette Score (Higher is Better)', color=color, fontweight='bold', fontsize=12)
ax2.plot(k_range, silhouette_list, marker='s', color=color, linewidth=2.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Optimal K Search on UMAP 10D Space (60k docs)', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

# 1. 10차원 성적표를 보고 마음의 결정을 내린 최종 K값 설정
final_k = 4  # 만약 10차원 지표를 보고 방을 5개로 늘리고 싶다면 이 숫자만 5로 바꾸세요!

print(f"🚀 [10차원 공간 기반] 확정된 K={final_k}로 최종 K-Means 군집화 가동...")
final_kmeans = KMeans(n_clusters=final_k, init='k-means++', random_state=42, n_init=10)

# ★ [수정 핵심 1] 꼬임 방지 통합 셀에서 만든 10차원 넘파이 배열(embeddings_umap_10d)을 지정합니다.
# ★ [수정 핵심 2] 컬럼명도 5차원과 헷갈리지 않게 'cluster_10d'로 깔끔하게 분리합니다.
df['cluster_10d'] = final_kmeans.fit_predict(embeddings_umap_10d)

print("\n📊 10차원 군집별 데이터 배정 개수 확인:")
print(df['cluster_10d'].value_counts())

# ★ [수정 핵심 3] 파일명에 10d를 명시하여 최종 결과물을 안전하게 분리 저장합니다.
output_filename = f"층간소음_군집완료_10d_{final_k}.pkl"
df.to_pickle(output_filename)

print(f"\n🎉 대장정 완료! 10차원 군집 결과가 반영된 데이터프레임이 '{output_filename}'로 안전하게 박제되었습니다.")

In [ ]:
#150차원, 5차원


import numpy as np
from umap import UMAP
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 시각화용 2차원 투영 데이터 생성 (기존 조건 유지)
if embeddings_umap.shape[1] > 2:
    print(f"🔄 현재 {embeddings_umap.shape[1]}차원 배열이므로, 시각화를 위해 2차원 UMAP 플롯을 추가 생성합니다...")
    umap_2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42, n_jobs=-1)
    viz_data = umap_2d.fit_transform(embeddings_umap)
else:
    viz_data = embeddings_umap

# --- 시각화 그래프 그리기 ---
# 범례(Legend)가 우측에 붙으므로 가로 너비를 12로 살짝 넓혀주는 것이 정석입니다.
plt.figure(figsize=(12, 9))
sns.set_theme(style='whitegrid')

# ★ [수정 핵심 포인트] ★
# color 지정을 빼고, hue에 방 번호 컬럼을 지정한 뒤 아름다운 팔레트를 입힙니다.
sns.scatterplot(
    x=viz_data[:, 0],
    y=viz_data[:, 1],
    hue=df['cluster'],  # 데이터프레임의 진짜 K-Means 방 번호 컬럼 입력
    palette='Set2',      # 아까 마음에 들어 하셨던 깔끔하고 예쁜 파스텔톤 색상 조합 (또는 'tab10')
    alpha=0.5,           # 6만 건의 겹침 밀도를 보기 위한 투명도 조절
    s=4,                 # 점의 크기 (대규모 데이터이므로 3~4가 가장 이쁩니다)
    linewidth=0          # ★중요: 점 테두리를 0으로 없애야 점들이 뭉쳤을 때 검은 테두리 때문에 색상이 칙칙해지지 않습니다.
)

# 그래프 제목 및 축 이름 레이블링
plt.title(f"BERT + PCA + UMAP 2D Final Cluster Map (K={df['cluster'].nunique()})", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("UMAP Dimension 1", fontsize=11)
plt.ylabel("UMAP Dimension 2", fontsize=11)

# 범례(어떤 색이 몇 번 방인지 설명하는 상자)를 그래프 우측 바깥으로 깔끔하게 빼줍니다.
plt.legend(title="Cluster No.", bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=3)

# 코랩에서 그래프를 이미지 파일로 고화질 저장 (발표 장표 백업용)
plt.savefig("umap_final_cluster_map.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"🎉 축하합니다! {df['cluster'].nunique()}개 방으로 선명하게 분할된 최종 색상 지도가 'umap_final_cluster_map.png'로 저장되었습니다!")

In [ ]:
!install pickle

In [ ]:
import pickle
# 1. UMAP으로 축소 완료한 넘파이 배열 저장 (pickle.dump 사용)
with open("embeddings_umap_5d.pkl", "wb") as f:
    pickle.dump(embeddings_umap, f)


print("✅ UMAP 배열 및 KMeans 모델 객체 피클 저장 완료!")

--------------------------
위에 이제 버려도 됨 여기부터 다시 시작


In [ ]:
import pandas as pd

# 1. 업로드한 5차원 피클 파일명을 적어줍니다 (파일명이 다르면 수정해 주세요!)
file_path = "층간소음_군집완료_4.pkl"

# 2. 판다스 내장 함수로 피클 안전하게 불러오기
df = pd.read_pickle(file_path)

print("🎉 5차원 군집 마스터 데이터프레임 로드 완료!")
print(f"📊 전체 데이터 개수: {len(df)}건\n")

# 3. 데이터가 원형 그대로 잘 들어왔는지 눈으로 확인
print("🔹 [확인 1] 데이터 상위 3개 행 확인:")
display(df.head(3))

print("\n🔹 [확인 2] 5차원 기반 군집별 배정 개수 다시 확인:")
if 'cluster' in df.columns:
    print(df['cluster'].value_counts())
else:
    # 혹시 컬럼명을 다른 이름으로 저장하셨다면 해당 컬럼명을 출력합니다.
    print("⚠️ 'cluster' 컬럼명을 찾을 수 없습니다. 현재 컬럼 목록을 확인하세요:")
    print(df.columns)

In [ ]:
import pickle

# UMAP 5차원 좌표 배열 피클 파일 불러오기
with open("embeddings_umap_5d.pkl", "rb") as f:
    embeddings_umap = pickle.load(f)

print("✅ 5차원 UMAP 좌표 배열 로드 완료! 모양(Shape):", embeddings_umap.shape)

In [ ]:
# 클러스터 3: 142개, 노이즈로 판단하고 제외하기 전 마지막 확인하기

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------------------------------------
# [준비 단계] token 컬럼이 리스트 형태라면 문자열로 합쳐줍니다.
# -------------------------------------------------------------
# 데이터프레임 이름은 df, 토큰 컬럼은 'token', 본문 컬럼은 '본문' 기준
if isinstance(df['token'].iloc[0], list):
    df['token_str'] = df['token'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
else:
    df['token_str'] = df['token'].fillna('')

# -------------------------------------------------------------
# [1단계] 군집별 고유 단어를 찾기 위한 대조군 TF-IDF 연산
# (4개 방의 텍스트를 각각 통째로 합쳐서 서로 비교 분석합니다)
# -------------------------------------------------------------
cluster_docs = df.groupby('cluster')['token_str'].apply(lambda x: ' '.join(x)).reset_index()

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(cluster_docs['token_str'])
feature_names = np.array(vectorizer.get_feature_names_out())

# 3번 군집의 행 인덱스 찾기
target_cluster = 3
target_row_idx = cluster_docs[cluster_docs['cluster'] == target_cluster].index[0]

# 3번 군집에서 TF-IDF 점수가 가장 높은 상위 20개 단어 추출
row_scores = tfidf_matrix.getrow(target_row_idx).toarray()[0]
top_word_indices = row_scores.argsort()[-20:][::-1]

print("=" * 60)
print(f"🚨 [검증 1] {target_cluster}번 군집을 대표하는 고유 TF-IDF 키워드 Top 20")
print("=" * 60)
for rank, idx in enumerate(top_word_indices, 1):
    if row_scores[idx] > 0:
        print(f"{rank:2d}등 | 단어: {feature_names[idx]:<10} | 점수: {row_scores[idx]:.4f}")

print("\n" + "=" * 60)
print(f"📄 [검증 2] {target_cluster}번 군집의 실제 본문 데이터 무작위 샘플 (15개)")
print("=" * 60)

# 3번 군집 글 중에서 15개를 뽑아 본문 내용을 확인합니다.
sample_texts = df[df['cluster'] == target_cluster]['본문'].dropna().sample(n=min(15, df[df['cluster'] == target_cluster].shape[0]), random_state=42)

for i, text in enumerate(sample_texts, 1):
    # 가독성을 위해 150자까지만 자르고, 줄바꿈은 공백으로 처리해서 출력
    clean_text = str(text).replace('\n', ' ').strip()
    print(f"[{i:2d}] {clean_text[:150]}...")
print("=" * 60)

# kmeans

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np

# 임베딩 데이터가 numpy 배열 형태인지 확인 (이미 embeddings 변수에 있다면 이대로 사용)
# 만약 df['embedding'] 컬럼만 있다면: embeddings = np.vstack(df['embedding'].values)

# 1. PCA를 통한 차원 축소 (768차원 -> 50차원)
print("PCA 차원 축소 진행 중...")
pca = PCA(n_components=50, random_state=42)
reduced_embeddings = pca.fit_transform(embeddings)
print("차원 축소 완료! 데이터 형태:", reduced_embeddings.shape)

# 2. 최적의 K값 찾기 (Elbow Method)
# 데이터가 5만 건이 넘으므로, K를 2~10까지 테스트해봅니다.
print("엘보우 그래프 생성 중...")
sse = []
k_range = range(2, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(reduced_embeddings)
    sse.append(kmeans.inertia_)

# 그래프 출력
plt.figure(figsize=(8, 5))
plt.plot(k_range, sse, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Sum of Squared Errors (SSE)')
plt.title('Elbow Method For Optimal K')
plt.show()

# ==========================================
# 3. K-Means 군집화 진행
# 위 그래프를 확인한 후, 꺾이는 지점(Elbow)의 K값을 아래 변수에 넣어주세요.
# ==========================================
optimal_k = 3
print(f"\nK={optimal_k}로 K-Means 최종 군집화 진행 중...")
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
cluster_labels = kmeans_final.fit_predict(reduced_embeddings)

# 4. 기존 데이터프레임에 군집 라벨(Cluster) 추가
df['cluster'] = cluster_labels
print("군집화 완료!")
display(df[['본문', 'token', 'cluster']].head(10))

In [ ]:
df

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import numpy as np

# 1. 메모리 에러 방지를 위해 1,000개 데이터 샘플링
sample_size = 1000
np.random.seed(42)

# 앞서 만든 50차원으로 축소된 reduced_embeddings 사용
sample_indices = np.random.choice(len(reduced_embeddings), sample_size, replace=False)
sample_data = reduced_embeddings[sample_indices]

# 2. 계층적 군집화 수행 (Ward 연결법: 군집 내 분산을 최소화하는 방식)
print("덴드로그램 계산 중 (샘플링 데이터 1,000개)...")
linked = linkage(sample_data, method='ward')

# 3. 덴드로그램 시각화
plt.figure(figsize=(15, 7))
dendrogram(linked,
           truncate_mode='lastp',  # 그래프가 너무 복잡해지는 것을 막기 위해 하위 노드 압축
           p=50,                   # 화면에 표시할 최대 군집(잎 노드)의 개수
           leaf_rotation=90.,
           leaf_font_size=10.,
           show_contracted=True)

plt.title('Hierarchical Clustering Dendrogram (Sampled 1,000 points)')
plt.xlabel('Cluster Size / Data Index')
plt.ylabel('Distance (Ward)')

# (선택) 적절한 거리(Distance)에서 선을 그어 군집이 나뉘는 모습 확인
# plt.axhline(y=10, color='r', linestyle='--') # y값은 출력된 그래프의 y축 스케일을 보고 조절하세요

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(25, 10)) # 가로 크기를 대폭 늘려서 세밀한 선들이 잘 보이게 함

dendrogram(linked,
           # truncate_mode='lastp',  # 이 두 줄을 주석 처리하면
           # p=50,                   # 1,000개 데이터의 전체 트리가 출력됩니다.
           truncate_mode='level',    # 대신 트리의 깊이(level)를 기준으로 보여주는 옵션을 쓸 수도 있습니다.
           p=10,                     # 위에서부터 10단계까지 내려가며 보여줌 (전체를 보려면 이 두 줄도 지우면 됩니다)
           leaf_rotation=90.,
           leaf_font_size=8.,
           show_contracted=True)

plt.title('Hierarchical Clustering Dendrogram (Detailed View)')
plt.xlabel('Data Index')
plt.ylabel('Distance (Ward)')

plt.tight_layout()
plt.show()

# clustering

In [ ]:
from sklearn.cluster import KMeans

# 1. k=3로 클러스터링 실행
kmeans3 = KMeans(n_clusters=3, init='k-means++', random_state=42)
df['cluster_k3'] = kmeans3.fit_predict(embeddings)


In [ ]:
print("--- [k=3] 군집별 분포 ---")
print(df['cluster_k3'].value_counts().sort_index())

In [ ]:
import pandas as pd

# 1. 중복 데이터 확인하기
# '통합본문' 열을 기준으로 완전히 똑같은 데이터가 몇 개인지 확인합니다.
duplicate_count = df.duplicated(subset=['본문']).sum()
print(f"🚨 현재 데이터프레임 내 중복된 텍스트 수: {duplicate_count}개")

In [ ]:
df_ = df.drop_duplicates(subset=['본문'], keep='first').reset_index(drop=True)

In [ ]:
df

In [ ]:
df = df.drop(columns=['cluster'])

In [ ]:
df

In [ ]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 토큰 결합 (띄어쓰기 기반 문자열로 변환)
def join_tokens(x):
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except:
            pass
    if isinstance(x, list):
        return " ".join(x)
    return str(x)

print("1. 토큰 결합 중...")
df['joined_tokens'] = df['token'].apply(join_tokens)

# 2. 군집별 텍스트 병합
print("2. 군집별 텍스트 병합 중...")
cluster_docs = df.groupby('cluster_k3')['joined_tokens'].apply(lambda x: " ".join(x)).reset_index()

# 3. TF-IDF 계산 (상위 단어가 풍부하게 뽑히도록 max_features를 5,000으로 확장)
print("3. TF-IDF 계산 중...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(cluster_docs['joined_tokens'])

feature_names = tfidf_vectorizer.get_feature_names_out()
dense_matrix = tfidf_matrix.todense()

# 4. 이미지와 동일한 2열 병렬 구조의 딕셔너리 빌딩
print("4. 결과 테이블 구축 중...")
result_table_dict = {}

for i in range(len(cluster_docs)):
    cluster_num = cluster_docs.iloc[i]['cluster_k3']

    # 현재 군집의 점수 매칭 및 정렬
    cluster_scores = dense_matrix[i].tolist()[0]
    word_scores = [(feature_names[j], cluster_scores[j]) for j in range(len(feature_names))]
    word_scores = sorted(word_scores, key=lambda x: x[1], reverse=True)[:40] # 상위 40개 추출

    # 단어 리스트와 점수 리스트로 각각 분리
    keywords = [ws[0] for ws in word_scores]
    scores = [ws[1] for ws in word_scores]

    # 이미지 속 컬럼명 규칙 매칭 (Cluster X Keyword, Cluster X Score)
    result_table_dict[f'Cluster {cluster_num} Keyword'] = keywords
    result_table_dict[f'Cluster {cluster_num} Score'] = scores

# 5. 최종 데이터프레임 생성 및 확인
tfidf_result_df = pd.DataFrame(result_table_dict)
print("🎯 TF-IDF 테이블 생성 완료!")
display(tfidf_result_df)

# (선택) 이 결과를 엑셀 파일로 바로 내보내서 크게 보고 싶다면 아래 주석을 해제하세요.
# tfidf_result_df.to_excel("cluster_tfidf_top40.xlsx", index=False)

In [ ]:
# ==========================================
# STEP 1: 원본 df에서 불용어 먼저 제거하기
# ==========================================
stop_words = {'하다', '있다', '되다', '보다'}

def clean_tokens(x):
    # 문자열 형태의 리스트라면 실제 리스트로 변환
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except:
            pass
    if isinstance(x, list):
        # 정의한 불용어에 포함되지 않는 단어만 남기기
        return [word for word in x if word not in stop_words]
    return x

print("1. 원본 df['token']에서 불용어 제거 중...")
df['token'] = df['token'].apply(clean_tokens)

# 토큰을 띄어쓰기 기반 문자열로 변환하여 새 컬럼에 저장
df['tokens'] = df['token'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

In [ ]:
df

# cluster 0 토픽 추출

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Cluster 0 데이터만 필터링하여 새로운 데이터프레임 생성
df_cluster0 = df[df['cluster_k3'] == 0].reset_index(drop=True)
print(f"🎯 Cluster 0 데이터 개수: {len(df_cluster0)}개")

# 2. Cluster 0의 임베딩 배열만 추출
X_c0 = np.stack(df_cluster0['embedding'].values)

# 3. 탐색할 하위 토픽 개수(K) 범위 설정
# 세부 토픽이므로 2~8개까지만 탐색합니다. (데이터 개수에 따라 조절 가능)
k_range_c0 = range(2, 9)

inertias_c0 = []
silhouette_scores_c0 = []

print("Cluster 0의 최적 하위 토픽 수(K)를 계산 중입니다...")

# 4. K값을 늘려가며 KMeans 학습 및 지표 계산
for k in k_range_c0:
    kmeans_c0 = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels_c0 = kmeans_c0.fit_predict(X_c0)

    # 엘보우 기법을 위한 Inertia(오차제곱합) 저장
    inertias_c0.append(kmeans_c0.inertia_)

    # 실루엣 점수 저장
    silhouette_scores_c0.append(silhouette_score(X_c0, labels_c0))

print("연산 완료! 결과를 그래프로 출력합니다.")

# 5. 시각화 (1행 2열)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# [왼쪽] 엘보우 기법 그래프
ax1.plot(k_range_c0, inertias_c0, marker='o', linestyle='-', color='blue')
ax1.set_title('Cluster 0: Elbow Method (Inertia)')
ax1.set_xlabel('Number of Sub-Topics (K)')
ax1.set_ylabel('Inertia')
ax1.set_xticks(k_range_c0)
ax1.grid(True)

# [오른쪽] 실루엣 점수 그래프
ax2.plot(k_range_c0, silhouette_scores_c0, marker='s', linestyle='-', color='red')
ax2.set_title('Cluster 0: Silhouette Score')
ax2.set_xlabel('Number of Sub-Topics (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(k_range_c0)
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cluster 0 데이터만 독립적인 데이터프레임으로 복사
df_c0 = df[df['cluster_k3'] == 0].copy().reset_index(drop=True)
X_c0 = np.stack(df_c0['embedding'].values)

# 2. K으로 세부 클러스터링 진행
n_cluster = 4
kmeans_c0 = KMeans(n_clusters=n_cluster, random_state=42, n_init='auto')
df_c0['sub_cluster'] = kmeans_c0.fit_predict(X_c0)

print("✨ Cluster 0의 세부 토픽(3개) 분리가 완료되었습니다!\n")
print("📊 [세부 토픽별 데이터 개수]")
print(df_c0['sub_cluster'].value_counts().sort_index())
print("\n==================================================")
print(" 🎯 [Cluster 0 하위 토픽 프로파일링: 키워드 + 대표 원문]")
print("==================================================")

# 3. c-TF-IDF를 위한 텍스트 병합 (이전 단계에서 만든 joined_tokens 활용)
cluster_docs_c0 = df_c0.groupby('sub_cluster')['joined_tokens'].apply(lambda x: " ".join(x)).reset_index()

tfidf_c0 = TfidfVectorizer(max_df=0.9, min_df=2)
tfidf_matrix_c0 = tfidf_c0.fit_transform(cluster_docs_c0['joined_tokens'])
feature_names_c0 = tfidf_c0.get_feature_names_out()

# 4. 각 하위 토픽별 핵심 단어(15개)와 중심 원문(2개) 추출
for i, sub_id in enumerate(cluster_docs_c0['sub_cluster']):
    print(f"\n▶️ [Sub-Topic {sub_id}]")

    # 핵심 단어 추출
    tfidf_scores = tfidf_matrix_c0[i].toarray()[0]
    top_indices = tfidf_scores.argsort()[::-1][:15]
    top_keywords = [feature_names_c0[idx] for idx in top_indices]
    print(f"🔑 핵심 단어: {', '.join(top_keywords)}")
    print("-" * 50)

    # 중심점 기반 대표 원문 추출
    sub_data = df_c0[df_c0['sub_cluster'] == sub_id].reset_index(drop=True)
    sub_embeddings = np.stack(sub_data['embedding'].values)
    centroid = sub_embeddings.mean(axis=0).reshape(1, -1)

    similarities = cosine_similarity(sub_embeddings, centroid).flatten()
    top_doc_indices = similarities.argsort()[::-1][:2]

    for rank, idx in enumerate(top_doc_indices):
        doc_text = sub_data.loc[idx, '본문']
        sim_score = similarities[idx]
        print(f"📄 [대표 원문 {rank+1} | 중심 유사도: {sim_score:.4f}]")
        print(f"{str(doc_text)[:250]}...\n") # 가독성을 위해 250자로 제한

# cluster 2 토픽 추출

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Cluster 2 데이터만 필터링하여 새로운 데이터프레임 생성
df_cluster2 = df[df['cluster_k3'] == 2].reset_index(drop=True)
print(f"🎯 Cluster 2 데이터 개수: {len(df_cluster2)}개")

# 2. Cluster 2의 임베딩 배열만 추출
X_c2 = np.stack(df_cluster2['embedding'].values)

# 3. 탐색할 하위 토픽 개수(K) 범위 설정
# 세부 토픽이므로 2~8개까지만 탐색합니다. (데이터 개수에 따라 조절 가능)
k_range_c2 = range(2, 9)

inertias_c2 = []
silhouette_scores_c2 = []

print("Cluster 2의 최적 하위 토픽 수(K)를 계산 중입니다...")

# 4. K값을 늘려가며 KMeans 학습 및 지표 계산
for k in k_range_c2:
    kmeans_c2 = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels_c2 = kmeans_c2.fit_predict(X_c2)

    # 엘보우 기법을 위한 Inertia(오차제곱합) 저장
    inertias_c2.append(kmeans_c2.inertia_)

    # 실루엣 점수 저장
    silhouette_scores_c2.append(silhouette_score(X_c2, labels_c2))

print("연산 완료! 결과를 그래프로 출력합니다.")

# 5. 시각화 (1행 2열)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# [왼쪽] 엘보우 기법 그래프
ax1.plot(k_range_c2, inertias_c2, marker='o', linestyle='-', color='blue')
ax1.set_title('Cluster 2: Elbow Method (Inertia)')
ax1.set_xlabel('Number of Sub-Topics (K)')
ax1.set_ylabel('Inertia')
ax1.set_xticks(k_range_c2)
ax1.grid(True)

# [오른쪽] 실루엣 점수 그래프
ax2.plot(k_range_c2, silhouette_scores_c2, marker='s', linestyle='-', color='red')
ax2.set_title('Cluster 2: Silhouette Score')
ax2.set_xlabel('Number of Sub-Topics (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(k_range_c2)
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cluster 2 데이터만 독립적인 데이터프레임으로 복사
df_c2 = df[df['cluster_k3'] == 2].copy().reset_index(drop=True)
X_c2 = np.stack(df_c2['embedding'].values)

# 2. K으로 세부 클러스터링 진행
n_cluster = 5
kmeans_c2 = KMeans(n_clusters=n_cluster, random_state=42, n_init='auto')
df_c2['sub_cluster'] = kmeans_c2.fit_predict(X_c2)

print("✨ Cluster 2의 세부 토픽(5개) 분리가 완료되었습니다!\n")
print("📊 [세부 토픽별 데이터 개수]")
print(df_c2['sub_cluster'].value_counts().sort_index())
print("\n==================================================")
print(" 🎯 [Cluster 2 하위 토픽 프로파일링: 키워드 + 대표 원문]")
print("==================================================")

# 3. c-TF-IDF를 위한 텍스트 병합 (이전 단계에서 만든 joined_tokens 활용)
cluster_docs_c2 = df_c2.groupby('sub_cluster')['joined_tokens'].apply(lambda x: " ".join(x)).reset_index()

tfidf_c2 = TfidfVectorizer(max_df=0.9, min_df=2)
tfidf_matrix_c2 = tfidf_c2.fit_transform(cluster_docs_c2['joined_tokens'])
feature_names_c2 = tfidf_c2.get_feature_names_out()

# 4. 각 하위 토픽별 핵심 단어(15개)와 중심 원문(2개) 추출
for i, sub_id in enumerate(cluster_docs_c2['sub_cluster']):
    print(f"\n▶️ [Sub-Topic {sub_id}]")

    # 핵심 단어 추출
    tfidf_scores = tfidf_matrix_c2[i].toarray()[0]
    top_indices = tfidf_scores.argsort()[::-1][:15]
    top_keywords = [feature_names_c2[idx] for idx in top_indices]
    print(f"🔑 핵심 단어: {', '.join(top_keywords)}")
    print("-" * 50)

    # 중심점 기반 대표 원문 추출
    sub_data = df_c2[df_c2['sub_cluster'] == sub_id].reset_index(drop=True)
    sub_embeddings = np.stack(sub_data['embedding'].values)
    centroid = sub_embeddings.mean(axis=0).reshape(1, -1)

    similarities = cosine_similarity(sub_embeddings, centroid).flatten()
    top_doc_indices = similarities.argsort()[::-1][:2]

    for rank, idx in enumerate(top_doc_indices):
        doc_text = sub_data.loc[idx, '본문']
        sim_score = similarities[idx]
        print(f"📄 [대표 원문 {rank+1} | 중심 유사도: {sim_score:.4f}]")
        print(f"{str(doc_text)[:250]}...\n") # 가독성을 위해 250자로 제한

In [ ]:
# 파일명 지정 (원하시는 이름으로 수정 가능합니다)
save_path = 'cluster2_sub_topics_5개로.pkl'

# df_c2 데이터프레임을 pickle 형식으로 저장
df_c2.to_pickle(save_path)

print(f"✅ Cluster 2 세부 토픽 데이터가 '{save_path}' 파일로 안전하게 저장되었습니다!")

In [ ]:
# 파일명 지정 (원하시는 이름으로 수정 가능합니다)
save_path = 'cluster0_sub_topics_4개로.pkl'

# df_c2 데이터프레임을 pickle 형식으로 저장
df_c0.to_pickle(save_path)

print(f"✅ Cluster 2 세부 토픽 데이터가 '{save_path}' 파일로 안전하게 저장되었습니다!")

# LDA

In [ ]:
!pip install pyLDAvis

In [ ]:
import pandas as pd

# 1. 통합본에 남기고 싶은 필수 칼럼(열) 리스트 정의
columns_to_keep = [
    '제목', '본문',
    'token',  'embedding',
    'cluster_k3', 'sub_cluster'
]

print("데이터프레임들을 하나로 합치는 중입니다...")

# 2. 각 데이터프레임에서 필요한 열만 필터링한 후, 하나의 리스트로 묶기
dfs_to_concat = [
    df_c0[columns_to_keep],
    df_c2[columns_to_keep]
]

# 3. 데이터프레임들을 위아래로 병합 (ignore_index=True로 기존 인덱스를 깔끔하게 리셋)
df_cluster_topic = pd.concat(dfs_to_concat, ignore_index=True)

# 4. 통합된 데이터 확인
print("✨ 4개의 세부 토픽 데이터가 하나로 완벽하게 통합되었습니다!")
print(f"📊 총 데이터 개수: {len(df_cluster_topic)}개")
print(f"✅ 남겨진 열(Columns): {list(df_cluster_topic.columns)}\n")

# 5. 임베딩 배열(embeddings) 유지를 위해 pickle 형식으로 최종 저장
save_path = 'df_cluster_topic.pkl'
df_cluster_topic.to_pickle(save_path)
print(f"💾 통합된 데이터프레임이 '{save_path}'로 안전하게 저장되었습니다.")

# 데이터프레임 상위 3줄 미리보기
display(df_cluster_topic.head(20))

In [ ]:
df['cluster_k3'].value_counts()

In [ ]:
!pip install pyLDAvis

In [ ]:
import gensim
from gensim import corpora, models
from gensim.corpora import Dictionary

In [ ]:
df_cluster0 = df[df['cluster_k3'] == 0]

all_documents = list(df_cluster0['token'])
# 고유한 단어들에 각각 고유한 정수 ID(번호)를 부여하여 사전 만들기
dictionary = Dictionary(all_documents)

In [ ]:
corpus = []
for i in all_documents:
  corpus.append(dictionary.doc2bow(i))

In [ ]:
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics = 3, id2word = dictionary, random_state = 42)

In [ ]:
align = []

for i in ldamodel.get_document_topics(corpus):
  label = []
  value = []
  for w in i:
    label.append(w[0])
    value.append(w[1])

  align.append(label[np.argmax(value)])

df_cluster0['topic'] = align
df_cluster0

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

prepared_data = gensimvis.prepare(ldamodel, corpus, dictionary) # 적용
pyLDAvis.save_html(prepared_data, 'ldavis_Action_0.html')
pyLDAvis.display(prepared_data)

In [ ]:
df_cluster0['topic'].value_counts()

In [ ]:
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics = 7, id2word = dictionary, random_state = 42)

In [ ]:
align = []

for i in ldamodel.get_document_topics(corpus):
  label = []
  value = []
  for w in i:
    label.append(w[0])
    value.append(w[1])

  align.append(label[np.argmax(value)])

df_cluster2['topic'] = align
df_cluster2

In [ ]:
!pip install pyLDAvis

In [ ]:
import gensim
from gensim import corpora, models
from gensim.corpora import Dictionary

In [ ]:
import pandas as pd
df = pd.read_pickle('/content/drive/MyDrive/DX PROJ./임베딩/이걸로/cluster2_sub_topics_5개로.pkl')

In [ ]:
df_cluster2 = df[df['cluster_k3'] == 2]

all_documents = list(df_cluster2['token'])
# 고유한 단어들에 각각 고유한 정수 ID(번호)를 부여하여 사전 만들기
dictionary = Dictionary(all_documents)

In [ ]:
corpus = []
for i in all_documents:
  corpus.append(dictionary.doc2bow(i))

In [ ]:
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics = 4, id2word = dictionary, random_state = 42)

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

prepared_data = gensimvis.prepare(ldamodel, corpus, dictionary) # 적용
pyLDAvis.save_html(prepared_data, 'ldavis_Action_2.html')
pyLDAvis.display(prepared_data)